In [1]:
import pandas as pd
import numpy as np
import cmdstanpy
from cmdstanpy import CmdStanModel

In [ ]:
from cmdstanpy import CmdStanModel 

# --- CONFIGURATION & MODEL DEPENDENCIES ---
STAN_FILE = "rl.stan"
TRANSITION_PROB_COMMON = 0.7 
# ------------------------------------------

# --- TEMPORARY TEST CONFIGURATION ---
TEST_SUBJECT_COUNT = 151 # Use only the first 151 subjects
TEST_TRIAL_COUNT = 200  # Use a maximum of 200 trials per subject
# ------------------------------------

# =============================================================================
# 1. LOAD & CLEAN DATA
# =============================================================================
print("Loading and cleaning data...")
try:
    df = pd.read_csv("final_dataset.csv")
except FileNotFoundError:
    print("❌ Error: 'final_dataset.csv' not found. Cannot proceed.")
    exit()

# Filter for stability and Stan compatibility
df = df[
    (df['rt_2'] >= 150) &    
    (df['choice_1'] != 0) &  
    (df['choice_2'] != 0) &  
    (df['state'] != 0)       # CRITICAL: Ensures no State 0 for indexing
]

# =============================================================================
# 2. SLICE DATA FOR SMALL TEST
# =============================================================================
unique_subjects = df['subject_id'].unique()
test_subjects = unique_subjects[:TEST_SUBJECT_COUNT] 

S = len(test_subjects)
T_max = TEST_TRIAL_COUNT 

print(f"--- RUNNING SMALL TEST ---")
print(f"Preparing matrices for {S} subjects with T_max={T_max}...")

# Initialize matrices based on the test T_max
c1_mat = np.zeros((S, T_max), dtype=int)
s2raw_mat = np.zeros((S, T_max), dtype=int)
c2_mat = np.zeros((S, T_max), dtype=int)
r_mat = np.zeros((S, T_max), dtype=float)
T_per_subject = []
prior_choice_vec = np.zeros(S, dtype=int)

# 3. FILL MATRICES WITH SLICED DATA
for i, subj in enumerate(test_subjects):
    subj_data = df[df['subject_id'] == subj]
    
    # Limit to TEST_TRIAL_COUNT
    subj_data = subj_data.head(TEST_TRIAL_COUNT) 
    
    n_trials = len(subj_data)
    T_per_subject.append(n_trials)
    
    # Fill the matrices row by row (up to n_trials, the rest remains 0-padded)
    c1_mat[i, :n_trials]    = subj_data['choice_1'].values.astype(int)
    s2raw_mat[i, :n_trials] = subj_data['state'].values.astype(int) 
    c2_mat[i, :n_trials]    = subj_data['choice_2'].values.astype(int)
    r_mat[i, :n_trials]     = subj_data['reward'].values.astype(float)
    
    # Get the very first choice for stickiness prior initialization
    prior_choice_vec[i] = subj_data['choice_1'].iloc[0]

# =============================================================================
# 4. FINAL STAN DATA DICTIONARY (with list conversion fix)
# =============================================================================

# CRITICAL FIX: Convert NumPy arrays to lists for robust CmdStanPy serialization
stan_data = {
    'S': S,
    'T_max': T_max,
    'T': T_per_subject, 
    'c1': c1_mat.tolist(),
    's2raw': s2raw_mat.tolist(), 
    'c2': c2_mat.tolist(),
    'r': r_mat.tolist(),
    'prior_choice': prior_choice_vec.tolist(),
    't_common': TRANSITION_PROB_COMMON, 
}

print(f"✅ Small test data dictionary created for {S} subjects x {T_max} max trials.")

# =============================================================================
# 5. RUNNING THE MODEL
# =============================================================================
# NOTE: This block assumes your Stan file and CmdStanPy setup are configured
model = CmdStanModel(stan_file=STAN_FILE)

fit = model.sample(
    data=stan_data,
    chains=4,
    iter_warmup=1000,    
    iter_sampling=1000,
    adapt_delta=0.95,   
    max_treedepth=12,
    inits=0,             
    show_progress=True,
    show_console=True
    )

summary = fit.summary()
print(summary.loc[[c for c in summary.index if 'alpha' in c or 'w_mb' in c]].head(5))

Loading and cleaning data...
--- RUNNING SMALL TEST ---
Preparing matrices for 151 subjects with T_max=200...


18:22:52 - cmdstanpy - INFO - Chain [1] start processing
18:22:52 - cmdstanpy - INFO - Chain [2] start processing
18:22:52 - cmdstanpy - INFO - Chain [3] start processing
18:22:52 - cmdstanpy - INFO - Chain [4] start processing


✅ Small test data dictionary created for 151 subjects x 200 max trials.

--- Attempting to run small test model: rl.stan ---
Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 1000 (Default)
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.95
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 12
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = 

19:58:43 - cmdstanpy - INFO - Chain [2] done processing


Chain [2] 
Chain [2] Elapsed Time: 3196.61 seconds (Warm-up)
Chain [2] 2552.19 seconds (Sampling)
Chain [2] 5748.81 seconds (Total)
Chain [2] 
Chain [2] 
Chain [1] Iteration: 2000 / 2000 [100%]  (Sampling)


...

Chain [1] 
Chain [1] Elapsed Time: 3240.77 seconds (Warm-up)
Chain [1] 2557.71 seconds (Sampling)
Chain [1] 5798.48 seconds (Total)
Chain [1] 
Chain [1] 
Chain [4] Iteration: 2000 / 2000 [100%]  (Sampling)


20:00:38 - cmdstanpy - INFO - Chain [4] done processing


Chain [4] 
Chain [4] Elapsed Time: 3276.68 seconds (Warm-up)
Chain [4] 2587.92 seconds (Sampling)
Chain [4] 5864.6 seconds (Total)
Chain [4] 
Chain [4] 
Chain [3] Iteration: 2000 / 2000 [100%]  (Sampling)


20:01:12 - cmdstanpy - INFO - Chain [3] done processing


Chain [3] 
Chain [3] Elapsed Time: 3438.75 seconds (Warm-up)
Chain [3] 2459.87 seconds (Sampling)
Chain [3] 5898.61 seconds (Total)
Chain [3] 
Chain [3] 

--- Sampling finished successfully! ---
Summary of parameter estimates for first 5 subjects:
              Mean      MCSE    StdDev       MAD        5%       50%  \
alpha[1]  0.399833  0.000795  0.070844  0.070142  0.285684  0.396180   
alpha[2]  0.620351  0.001832  0.115675  0.116662  0.438112  0.617026   
alpha[3]  0.342250  0.001842  0.103845  0.097443  0.199644  0.329985   
alpha[4]  0.807120  0.001265  0.084486  0.087624  0.663349  0.809326   
alpha[5]  0.157624  0.001670  0.097393  0.096446  0.028448  0.145903   

               95%  ESS_bulk  ESS_tail  ESS_bulk/s     R_hat  
alpha[1]  0.515653   8078.23   2757.01    0.795282  1.000210  
alpha[2]  0.819174   4121.03   2230.64    0.405705  0.999842  
alpha[3]  0.527990   3715.78   2667.85    0.365810  1.002660  
alpha[4]  0.942610   3987.12   1869.60    0.392523  1.000950  
alph

In [39]:
import numpy as np
import pandas as pd

# --- CORRECTION ---
# Use fit.stan_variable() to get the draws as a NumPy array (draws x S x T_max)
y1_rep_all = fit.stan_variable('y1_rep')
y2_rep_all = fit.stan_variable('y2_rep')

# N_draws is calculated from the shape of the resulting array
N_draws = y1_rep_all.shape[0]

# --- IMPORTANT: Ensure c1_mat and c2_mat are available and contain 1s and 2s ---
# Assuming c1_mat and c2_mat are the original NumPy arrays (S x T_max)
# If they are not available in the current scope, you must load them first!

# --- 1. Calculate Accuracy for Stage 1 (c1) ---

# Observed choices (reshaped to match draws' dimensions for comparison)
# Replicate the observed data N_draws times.
c1_observed_tiled = np.tile(c1_mat, (N_draws, 1, 1))

# Compare replicated choices to observed choices
stage1_matches = (y1_rep_all == c1_observed_tiled).astype(int)

# Use the T array to mask out padded trials (where T[s] < T_max)
S = stan_data['S']
T_max = stan_data['T_max']
T_per_subject = stan_data['T']

mask = np.zeros((S, T_max), dtype=bool)
for s in range(S):
    mask[s, :T_per_subject[s]] = True
mask_tiled = np.tile(mask, (N_draws, 1, 1))

# Apply the mask to the matches
stage1_matches_valid = stage1_matches[mask_tiled]

# Calculate the mean accuracy across all valid trials and all posterior draws
mean_accuracy_stage1 = np.mean(stage1_matches_valid)

# --- 2. Calculate Accuracy for Stage 2 (c2) ---
c2_observed_tiled = np.tile(c2_mat, (N_draws, 1, 1))
stage2_matches = (y2_rep_all == c2_observed_tiled).astype(int)
stage2_matches_valid = stage2_matches[mask_tiled]
mean_accuracy_stage2 = np.mean(stage2_matches_valid)

print(f"\n--- Posterior Predictive Accuracy (Across All Subjects & Trials) ---")
print(f"Stage 1 (c1) Predictive Accuracy: {mean_accuracy_stage1:.4f} ({mean_accuracy_stage1*100:.2f}%)")
print(f"Stage 2 (c2) Predictive Accuracy: {mean_accuracy_stage2:.4f} ({mean_accuracy_stage2*100:.2f}%)")


--- Posterior Predictive Accuracy (Across All Subjects & Trials) ---
Stage 1 (c1) Predictive Accuracy: 0.6875 (68.75%)
Stage 2 (c2) Predictive Accuracy: 0.6448 (64.48%)


In [40]:
import numpy as np

# --- 1. Load Replicated Data ---
# Use fit.stan_variable() to get the draws as a NumPy array (draws x S x T_max)
y1_rep_all = fit.stan_variable('y1_rep')
y2_rep_all = fit.stan_variable('y2_rep')
N_draws = y1_rep_all.shape[0]

# --- 2. Setup Masking and Observed Data ---
S = stan_data['S']
T_max = stan_data['T_max']
T_per_subject = stan_data['T']

# Create a mask for valid (non-padded) trials
mask = np.zeros((S, T_max), dtype=bool)
for s in range(S):
    mask[s, :T_per_subject[s]] = True
mask_tiled = np.tile(mask, (N_draws, 1, 1))

# Replicate the observed choice matrices
c1_observed_tiled = np.tile(c1_mat, (N_draws, 1, 1))
c2_observed_tiled = np.tile(c2_mat, (N_draws, 1, 1))

# --- 3. Calculate RMSE for Stage 1 (c1) ---

# Calculate the squared error (will be 0 for match, 1 for mismatch)
squared_error_stage1 = (y1_rep_all - c1_observed_tiled)**2

# Apply the mask and calculate Mean Squared Error (MSE)
squared_error_stage1_valid = squared_error_stage1[mask_tiled]
mse_stage1 = np.mean(squared_error_stage1_valid)
rmse_stage1 = np.sqrt(mse_stage1)

# --- 4. Calculate RMSE for Stage 2 (c2) ---
squared_error_stage2 = (y2_rep_all - c2_observed_tiled)**2
squared_error_stage2_valid = squared_error_stage2[mask_tiled]
mse_stage2 = np.mean(squared_error_stage2_valid)
rmse_stage2 = np.sqrt(mse_stage2)

print(f"\n--- Root Mean Square Error (RMSE) ---")
print(f"Stage 1 (c1) RMSE: {rmse_stage1:.4f}")
print(f"Stage 2 (c2) RMSE: {rmse_stage2:.4f}")


--- Root Mean Square Error (RMSE) ---
Stage 1 (c1) RMSE: 0.5590
Stage 2 (c2) RMSE: 0.5960


In [41]:
import pandas as pd

# Load all parameter draws into a DataFrame. 
# This DataFrame will have (chains * draws) rows and many columns (one for each parameter).
all_draws_df = fit.draws_pd()

# Display the first few rows and columns to show the structure
print("\n--- All Posterior Draws DataFrame ---")
print(all_draws_df.head())
print(f"Shape of Draws DataFrame: {all_draws_df.shape}")

# Example: Save the full raw data to a CSV file
# You MUST save this file as it contains all the results!
all_draws_df.to_csv("hybrid_rl_posterior_draws.csv", index=False)
print("✅ Saved all posterior draws to hybrid_rl_posterior_draws.csv")


--- All Posterior Draws DataFrame ---
   chain__  iter__  draw__       lp__  accept_stat__  stepsize__  treedepth__  \
0      1.0     1.0     1.0 -29180.591       0.929866    0.073445          6.0   
1      1.0     2.0     2.0 -29158.944       0.994828    0.073445          6.0   
2      1.0     3.0     3.0 -29205.451       0.965648    0.073445          6.0   
3      1.0     4.0     4.0 -29125.440       0.965732    0.073445          6.0   
4      1.0     5.0     5.0 -29115.277       0.958958    0.073445          6.0   

   n_leapfrog__  divergent__   energy__  ...  y2_rep[142,200]  \
0          63.0          0.0  29624.114  ...              1.0   
1          63.0          0.0  29650.032  ...              1.0   
2          63.0          0.0  29651.901  ...              1.0   
3          63.0          0.0  29594.310  ...              1.0   
4          63.0          0.0  29550.618  ...              1.0   

   y2_rep[143,200]  y2_rep[144,200]  y2_rep[145,200]  y2_rep[146,200]  \
0         

In [42]:
# Get the full summary table (Mean, StdDev, R_hat, ESS, etc.)
full_summary_df = fit.summary()

# Save the full summary table
full_summary_df.to_csv("hybrid_rl_parameter_summary.csv")
print("✅ Saved full parameter summary to hybrid_rl_parameter_summary.csv")

✅ Saved full parameter summary to hybrid_rl_parameter_summary.csv


In [43]:
import pandas as pd

# Get the full summary table containing all parameters and generated quantities
full_summary_df = fit.summary()

# Define the base names of the parameters and output variables you want to keep
rl_parameter_bases = [
    'alpha', 
    'lambda_', 
    'w_mb', 
    'beta1', 
    'beta2', 
    'stickiness', 
    'log_lik'
]

# Create a boolean mask to filter the DataFrame index
# The '|' (OR) operator is used to combine the conditions
mask = pd.Series([False] * len(full_summary_df), index=full_summary_df.index)

for base in rl_parameter_bases:
    # Check if the parameter name starts with the base name (e.g., 'alpha[1]')
    mask = mask | full_summary_df.index.str.startswith(base)

# Filter the DataFrame to keep only the selected parameters
filtered_rl_summary_df = full_summary_df[mask]

# Display the first few rows of the filtered results
print("\n--- Filtered RL Parameter Summary (Mean, R_hat, ESS) ---")
print(filtered_rl_summary_df.head(10)) 

# Save the filtered results to a new CSV file
filtered_rl_summary_df.to_csv("core_rl_parameter_summary_filtered.csv")
print("✅ Saved filtered RL parameter summary to core_rl_parameter_summary_filtered.csv")

# --- Optional: Total Log-Likelihood ---
# The total log-likelihood is useful for model comparison (WAIC/LOO).
# It's the sum of all log_lik[s,t] values.
total_log_lik = filtered_rl_summary_df.loc[filtered_rl_summary_df.index.str.startswith('log_lik'), 'Mean'].sum()
print(f"\nTotal Mean Log-Likelihood (Sum of all log_lik[s,t] means): {total_log_lik:.2f}")


--- Filtered RL Parameter Summary (Mean, R_hat, ESS) ---
               Mean      MCSE    StdDev       MAD        5%       50%  \
alpha[1]   0.399833  0.000795  0.070844  0.070142  0.285684  0.396180   
alpha[2]   0.620351  0.001832  0.115675  0.116662  0.438112  0.617026   
alpha[3]   0.342250  0.001842  0.103845  0.097443  0.199644  0.329985   
alpha[4]   0.807120  0.001265  0.084486  0.087624  0.663349  0.809326   
alpha[5]   0.157624  0.001670  0.097393  0.096446  0.028448  0.145903   
alpha[6]   0.196525  0.026020  0.251415  0.023421  0.008808  0.026932   
alpha[7]   0.730075  0.000706  0.059763  0.059410  0.630637  0.730972   
alpha[8]   0.616742  0.000788  0.068780  0.069148  0.505759  0.614877   
alpha[9]   0.020089  0.001043  0.047312  0.008862  0.001003  0.009549   
alpha[10]  0.526789  0.001211  0.086057  0.087386  0.386394  0.524647   

                95%  ESS_bulk  ESS_tail  ESS_bulk/s     R_hat  
alpha[1]   0.515653  8078.230  2757.010    0.795282  1.000210  
alpha[2]  

In [45]:
import numpy as np

# Select the R_hat column from the filtered summary DataFrame
rhat_values = filtered_rl_summary_df['R_hat']

# Find the key summary statistics for R_hat
min_rhat = rhat_values.min()
max_rhat = rhat_values.max()
mean_rhat = rhat_values.mean()
n_rhat_above_1_01 = (rhat_values > 1.01).sum()

print("\n--- Overall R-hat Diagnostics (Across All Core RL Parameters) ---")
print(f"Total number of core parameters checked: {len(rhat_values)}")
print(f"Minimum R-hat: {min_rhat:.5f}")
print(f"Maximum R-hat: {max_rhat:.5f}")
print(f"Mean R-hat: {mean_rhat:.5f}")
print(f"Number of parameters with R-hat > 1.01: {n_rhat_above_1_01}")



--- Overall R-hat Diagnostics (Across All Core RL Parameters) ---
Total number of core parameters checked: 31106
Minimum R-hat: 0.99917
Maximum R-hat: 1.04236
Mean R-hat: 1.00094
Number of parameters with R-hat > 1.01: 155


In [47]:
import numpy as np

# --- ESS Diagnostic Code ---

# Select the ESS columns from the filtered summary DataFrame
ess_bulk_values = filtered_rl_summary_df['ESS_bulk']
ess_tail_values = filtered_rl_summary_df['ESS_tail']

# The total number of posterior draws (4 chains * 400 sampling iterations)
TOTAL_POSTERIOR_DRAWS = 4 * 400 

# --- ESS_bulk (Reliability of Mean/Median) ---
min_ess_bulk = ess_bulk_values.min()
max_ess_bulk = ess_bulk_values.max()
mean_ess_bulk = ess_bulk_values.mean()

# --- ESS_tail (Reliability of Credible Intervals) ---
min_ess_tail = ess_tail_values.min()
max_ess_tail = ess_tail_values.max()
mean_ess_tail = ess_tail_values.mean()

print("\n--- Overall ESS Diagnostics (Across All Core RL Parameters) ---")
print(f"Total posterior draws (N): {TOTAL_POSTERIOR_DRAWS}")
print(f"Total number of core parameters checked: {len(ess_bulk_values)}")


print("\n### ESS_bulk (Main Body of Posterior) ###")
print(f"Minimum ESS_bulk: {min_ess_bulk:.1f}")
print(f"Maximum ESS_bulk: {max_ess_bulk:.1f}")
print(f"Mean ESS_bulk: {mean_ess_bulk:.1f}")
print(f"Minimum ESS_bulk / N Ratio: {min_ess_bulk / TOTAL_POSTERIOR_DRAWS:.4f}")

print("\n### ESS_tail (Credible Intervals) ###")
print(f"Minimum ESS_tail: {min_ess_tail:.1f}")
print(f"Maximum ESS_tail: {max_ess_tail:.1f}")
print(f"Mean ESS_tail: {mean_ess_tail:.1f}")



--- Overall ESS Diagnostics (Across All Core RL Parameters) ---
Total posterior draws (N): 1600
Total number of core parameters checked: 31106

### ESS_bulk (Main Body of Posterior) ###
Minimum ESS_bulk: 101.0
Maximum ESS_bulk: 11481.7
Mean ESS_bulk: 5716.8
Minimum ESS_bulk / N Ratio: 0.0631

### ESS_tail (Credible Intervals) ###
Minimum ESS_tail: 502.1
Maximum ESS_tail: 4111.2
Mean ESS_tail: 3036.3
